目的/objective
* PyTorchでCNNを構築しMNISTの画像分類を行う。
* Build a CNN with PyTorch and perform image classification on MNIST.

準備/Preparation
* Graphics Processing Unit (GPU)を用いて処理を行うために、上部のメニューバーの「ランタイム」→「ランタイムのタイプを変更」からハードウェアアクセラレータをGPUにしてください。
* To perform processing using a Graphics Processing Unit (GPU), please go to the top menu bar, select "Runtime" → "Change runtime type", and set the Hardware accelerator to GPU.

In [1]:
from time import time   # 時間計測を行うモジュール/A module for time measurement

import numpy as np      # Pythonで配列を扱うためのモジュール/Module for handling arrays in Python
import torch            # 深層学習のモジュール（PyTorch）/Deep learning module (PyTorch)
import torch.nn as nn

import torchvision      # PyTorchで画像認識に使用するネットワークやデータセットを利用するためのモジュール/Modules for using networks and datasets for image recognition in PyTorch
import torchvision.transforms as transforms

import torchsummary     # PyTorchで作成したネットワークモデルの詳細を表示するモジュール/Module for displaying details of a neural network model created with PyTorch

* GPUを使用した計算が可能かどうかを確認します。
* `Use CUDA: True`と表示されればGPUを使用した計算をPyTorchで行うことが可能です。
Falseとなっている場合は上記の「Google Colaboratoryの設定確認・変更」に記載している手順にしたがって設定を変更した後にモジュールのインポートから始めてください。
---
* Check whether calculations using the GPU are possible.
* If Use CUDA: True is displayed, it means you can perform computations using the GPU in PyTorch.
If it shows False, follow the steps described in “Checking and Changing Settings in Google Colaboratory” above, and then start again from importing the modules.

In [2]:
use_cuda = torch.cuda.is_available()
print('Use CUDA:', use_cuda)

Use CUDA: True


学習データ（MNIST Dataset）を読み込みます。
* 読み込んだ学習データのサイズを確認
* 学習データ数は6万枚、評価データは1万枚、1つのデータのサイズは28x28の786次元

`torchvision.datasets.MNIST()`を用いてMNISTデータセットを自動的にダウンロードして読み込みます。

引数（以下）
* root：MNISTデータの保存場所
* train：学習またはテストデータの指定（Trueの場合に学習用データを扱う）
* transform：画像データに対する処理（詳細は後述）
* download：データが`root`に存在しない場合，ウェブからダウンロードするか
---
Load the training data (MNIST Dataset).
* Check the size of the loaded training data.
* The training dataset consists of 60,000 images, the evaluation dataset consists of 10,000 images, and each data sample has a size of 28×28, which is 784 dimensions.

Automatically download and load the MNIST dataset using torchvision.datasets.MNIST().

argument (below)
* root：Where MNIST data is stored
* train：Specify training or test data (if True, training data will be used)
* transform：Image data processing (details below)
* download：If the data does not exist in root, download it from the web.

In [3]:
train_data = torchvision.datasets.MNIST(root="./", train=True, transform=transforms.ToTensor(), download=True)
test_data = torchvision.datasets.MNIST(root="./", train=False, transform=transforms.ToTensor(), download=True)

### 読み込んだデータの情報を表示して確認します/Display and check the information of the loaded data
# 画像・ラベルデータのデータタイプ（型）/Data types (formats) of image and label data
print(type(train_data.data), type(train_data.targets))
print(type(test_data.data), type(test_data.targets))
# 画像・ラベルの配列サイズ/Array size of images and labels
print(train_data.data.size(), train_data.targets.size())
print(test_data.data.size(), test_data.targets.size())

100%|██████████| 9.91M/9.91M [00:01<00:00, 4.99MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.3MB/s]

<class 'torch.Tensor'> <class 'torch.Tensor'>
<class 'torch.Tensor'> <class 'torch.Tensor'>
torch.Size([60000, 28, 28]) torch.Size([60000])
torch.Size([10000, 28, 28]) torch.Size([10000])


このネットワークは畳み込み層2層と全結合層3層で構成されるCNNです。

* 畳み込み層

  1層目：入力チャンネル1、出力16、フィルタサイズ3×3

  2層目：入力16、出力32、フィルタサイズ3×3

  畳み込み後はシグモイド関数で活性化し、max poolingでプーリングします。

* 全結合層

  1層目：入力7×7×32、出力1024

  2層目：入力1024、出力1024

  出力層：入力1024、出力10（クラス数）

* 順伝播（forward）処理の流れ

  1. 入力データを1層目の畳み込み層に通す → 活性化 → プーリング
  
  2. 2層目も同様に畳み込み → 活性化 → プーリング
  
  3. 特徴マップを1次元に変換（view(h.size()[0], -1)）
  
  4. 全結合層に順次入力し、最終的にクラススコアを出力
---
This network is a CNN composed of two convolutional layers and three fully connected layers.

* convolutional layer

  first layer：Input channels: 1, Output channels: 16, Filter size: 3×3

  second layer：Input 16, output 32, filter size 3×3

  After convolution, the activation is performed using the sigmoid function, followed by max pooling.

* fully connected layer

  first layer：Input 7×7×32, output 1024

  second layer：Input 1024, Output 1024

  output layer：Input: 1024, Output: 10 (number of classes)

* Flow of forward propagation processing

  1. Pass the input data through the first convolutional layer → Activation → Pooling
  
  2. Perform convolution on the second layer in the same way → Activation → Pooling
  
  3. Flatten the feature map into one dimension（view(h.size()[0], -1)）
  
  4. Sequentially input into the fully connected layers, ultimately producing the class scores

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.l1 = nn.Linear(7*7*32, 1024)
        self.l2 = nn.Linear(1024, 1024)
        self.l3 = nn.Linear(1024, 10)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):                       # 入力画像サイズ/input image size: [batch, 1, 28, 28]
        h = self.pool(self.act(self.conv1(x)))  # 出力される特徴マップサイズ/output feature map size: [batch, 16, 14, 14]
        h = self.pool(self.act(self.conv2(h)))  # 出力される特徴マップサイズ/output feature map size: [batch, 32, 7, 7]
        # 特徴マップの配列サイズを [batch, channel, height, width] --> [batch, channel * height * width] に変更/Resize the feature map array
        h = h.view(h.size()[0], -1)
        h = self.act(self.l1(h))
        h = self.act(self.l2(h))
        h = self.l3(h)
        return h

* CNNクラスを呼び出してネットワークモデルを作成し、use_cuda が True の場合はGPU上に配置して高速演算を可能にします。学習にはモーメンタム付き確率的勾配降下法（SGD）を使用し、学習率0.01、モーメンタム0.9で最適化します。最後に torchsummary.summary() を使ってネットワークの構造を表示します。
* The CNN class is called to create the network model, and if use_cuda is True, the model is placed on the GPU for faster computation. The model is optimized using momentum stochastic gradient descent (SGD) with a learning rate of 0.01 and momentum of 0.9. Finally, the network architecture is displayed using torchsummary.summary().

In [5]:
### ネットワークの作成/Create a network
model = CNN()
if use_cuda:      # GPUが使用できる場合は/If a GPU is available
    model.cuda()  # ネットワークの演算をGPUで行うように設定変更 (デフォルトはCPU)/Change the settings to perform network computations on the GPU (default is CPU)

### 最適化手法の設定 lr: 学習率, momentum: モーメンタム (慣性項のパラメータ)/Optimizer settings: lr (learning rate), momentum (momentum, the parameter of the inertia term)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

### 定義したモデルの情報を表示/Display information of the defined model
if use_cuda:
    torchsummary.summary(model, (1, 28, 28), device='cuda')  # GPUで演算する設定の場合/When set to perform computations on the GPU
else:
    torchsummary.summary(model, (1, 28, 28), device='cpu')   # CPUで演算する設定の場合/When set to compute on the CPU

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 28, 28]             160
              ReLU-2           [-1, 16, 28, 28]               0
         MaxPool2d-3           [-1, 16, 14, 14]               0
            Conv2d-4           [-1, 32, 14, 14]           4,640
              ReLU-5           [-1, 32, 14, 14]               0
         MaxPool2d-6             [-1, 32, 7, 7]               0
            Linear-7                 [-1, 1024]       1,606,656
              ReLU-8                 [-1, 1024]               0
            Linear-9                 [-1, 1024]       1,049,600
             ReLU-10                 [-1, 1024]               0
           Linear-11                   [-1, 10]          10,250
Total params: 2,671,306
Trainable params: 2,671,306
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forw

MNISTデータセットと作成したネットワークを使って学習を行います。

ミニバッチサイズは100、学習エポック数は10とします。

1. データローダーの定義

    データローダーでtrain_dataをミニバッチ単位で読み込みます。shuffle=Trueにすることで、毎回データの順序をランダムにします。

2. 誤差関数の設定

    分類問題のため、クロスエントロピー誤差をCrossEntropyLossとして定義します。

3. 学習の実行

    各ミニバッチに対して、

  * 入力画像imageをモデルに与え、各クラスの確率yを取得
  * 出力yと教師ラベルtから誤差を計算
  * 認識精度も算出
  * backward()で誤差を逆伝播し、ネットワークのパラメータを更新
---
Train a network using the MNIST dataset.

The mini-batch size is 100, and the number of epochs is 10.

1. Define the DataLoader

    The DataLoader loads train_data in mini-batches. Setting shuffle=True randomizes the data order each time.

2. Set the Loss Function

    For a classification task, we define the cross-entropy loss as CrossEntropyLoss.

3. Training Process

    For each mini-batch:

  * Feed input images image into the model to get class probabilities y
  * Compute the loss between output y and target labels t
  * Calculate the accuracy
  * Backpropagate the loss using backward() and update the network parameters

In [6]:
# ミニバッチサイズ・エポック数の設定/Setting the mini-batch size and number of epochs
batch_size = 100
epoch_num = 10

# データローダーの設定/Data loader settings
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)

# 誤差関数の設定/Error function setup
criterion = nn.CrossEntropyLoss()
if use_cuda:
    criterion.cuda()

# ネットワークを学習モードへ変更/Set the network to training mode
model.train()

# 学習の実行/Execute learning
train_start = time()
for epoch in range(1, epoch_num+1):   # epochのforループ/for loop over epochs
    # 1 epochの学習中の誤差・学習画像が正解した数をカウントする変数を初期化/Initialize variables to count the training error and the number of correctly classified training images during one epoch
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:  # 1 epoch内のforループ (iterationのループ)/for loop within one epoch (iteration loop)

        if use_cuda:  # GPUで計算する場合はデータもGPUメモリ上へ移動させる/When performing computations on a GPU, the data must also be transferred to the GPU memory
            image = image.cuda()
            label = label.cuda()

        y = model(image)  # データの入力と結果の出力/Inputting data and outputting results

        # 誤差計算とbackpropagation, パラメータの更新/Error calculation, backpropagation, and parameter update
        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        # 学習経過を確認するための処理/Process to monitor training progress
        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    # 1 epoch終了時点での誤差の平均値，学習データに対する認識精度, 学習開始からの経過時間を表示/Display the average error at the end of 1 epoch, the recognition accuracy on the training data, and the elapsed time since the start of training
    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(epoch, sum_loss/len(train_loader), count.item()/len(train_data), time() - train_start))

epoch: 1, mean loss: 0.5847234443326791, mean accuracy: 0.8239166666666666, elapsed time: 9.324814081192017
epoch: 2, mean loss: 0.1035838923137635, mean accuracy: 0.9676, elapsed time: 17.743597745895386
epoch: 3, mean loss: 0.0675721250846982, mean accuracy: 0.97885, elapsed time: 26.51874303817749
epoch: 4, mean loss: 0.05030888208343337, mean accuracy: 0.9838666666666667, elapsed time: 35.307618618011475
epoch: 5, mean loss: 0.03983804398061087, mean accuracy: 0.9874666666666667, elapsed time: 44.691734790802
epoch: 6, mean loss: 0.03170185854251031, mean accuracy: 0.9896166666666667, elapsed time: 52.885613441467285
epoch: 7, mean loss: 0.026239222771061273, mean accuracy: 0.9916333333333334, elapsed time: 62.325355052948
epoch: 8, mean loss: 0.02122177014690048, mean accuracy: 0.99335, elapsed time: 71.27848863601685
epoch: 9, mean loss: 0.01747470542099715, mean accuracy: 0.9944833333333334, elapsed time: 79.61300992965698
epoch: 10, mean loss: 0.014715262211587591, mean accurac

学習したネットワークを用いてテストデータに対する認識率の確認を行います。

* `model.eval()`を適用することでネットワーク演算を評価モードへ変更します。
これにより学習時と評価時で挙動が異なる演算（dropout等）を変更することが可能です。
* また`torch.no_grad()`を適用することで学習時には必要になる勾配情報を保持することなく演算を行います。
---
Evaluate the recognition accuracy on the test data using the trained network.

* Applying model.eval() switches the network operations to evaluation mode.
This allows operations that behave differently during training and evaluation (such as dropout) to be modified accordingly.
* By applying torch.no_grad(), computations can be performed without retaining gradient information that would be needed during training.

In [7]:
# データローダーの準備/Preparing the data loader
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

# ネットワークを評価モードへ変更/Set network to evaluation mode
model.eval()

# 評価の実行/Perform evaluation
count = 0
with torch.no_grad():  # 勾配を計算しない設定にする (loss.backwardをしても勾配情報が計算されない)/Set to not compute gradients (even if loss.backward is called, gradient information will not be computed)
    for image, label in test_loader:

        if use_cuda:
            image = image.cuda()
            label = label.cuda()

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / 10000.))

test accuracy: 0.9884


**課題**

##### 1. ネットワークの構造を変更し認識精度の変化を確認しましょう。

**ヒント：ネットワーク構造の変更としては次のようなものが考えられます**
* 中間層のユニット数
* 層の数
* 活性化関数
  * `nn.Tanh()`や`nn.ReLU()`, `nn.LeakyReLU()`などが考えられます。
  * その他のPyTorchで使用できる活性化関数は[こちらページ](https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity)にまとめられています。

※ ネットワーク構造を変更した際には`torchsummary.summary(***)`を使用しネットワーク構造を変更した際のパラメータ数の変化を確認してみましょう。

##### 2. 学習の設定を変更し認識精度の変化を確認しましょう。

**ヒント：プログラムの中で変更で切る設定は次のようなものが存在します**
* ミニバッチサイズ
* 学習回数（Epoch数）
* 学習率
* 最適化手法
  * `torch.optim.Adagrad()`や`torch.optim.Adam()`などが考えられます。
  * PyTorchで使用できる最適化手法は[こちらのページ](https://pytorch.org/docs/stable/optim.html#algorithms)にまとめられています。